# Hash Tables

*Map keys to storage buckets, resolve collisions, and update an existing key.*

A **hash table** stores key-value pairs and uses a **hash function** to
choose where to look for a key. Python dictionaries provide hash-based
lookup. The small class here exposes the main idea with a list of buckets.


## Hashing and Collisions

`hash(key) % size` converts a key's hash to a bucket index. Different keys
can produce the same index: this is a **collision**. **Chaining** handles
a collision by storing several pairs in one bucket and checking their keys.

With ten buckets, the positive integer IDs `11`, `21`, and `31` all select
bucket 1. Integer keys make this small example's bucket layout repeatable.

```text
11 -- hash % 10 --+
21 -- hash % 10 --+--> bucket 1: [(11, 100), (21, 200), (31, 300)]
31 -- hash % 10 --+
```

`put()` replaces a value when the key already exists and appends a new
pair otherwise. `get()` checks the selected bucket and raises `KeyError`
if the key is absent. This demonstrates chaining; Python's dictionary
uses a different internal collision-handling design.


In [1]:
class HashTable:
    def __init__(self, size=10):
        self.size = size
        self.buckets = [[] for _ in range(size)]

    def _bucket_index(self, key):
        return hash(key) % self.size

    def put(self, key, value):
        bucket = self.buckets[self._bucket_index(key)]
        for position, (stored_key, stored_value) in enumerate(bucket):
            if stored_key == key:
                bucket[position] = (key, value)
                return
        bucket.append((key, value))

    def get(self, key):
        bucket = self.buckets[self._bucket_index(key)]
        for stored_key, stored_value in bucket:
            if stored_key == key:
                return stored_value
        raise KeyError(key)


study_minutes = HashTable()
for student_id, minutes in [(11, 100), (21, 200), (31, 300)]:
    study_minutes.put(student_id, minutes)

print("Bucket for ID 11:", study_minutes._bucket_index(11))
print("Bucket 1:", study_minutes.buckets[1])
print("Minutes for ID 21:", study_minutes.get(21))

Bucket for ID 11: 1
Bucket 1: [(11, 100), (21, 200), (31, 300)]
Minutes for ID 21: 200


All three pairs remain in bucket 1. Looking up `21` returns `200` because
the lookup checks the stored key as well as the bucket index. A collision
does not mean that two keys are equal.


## Updating an Existing Key

Key uniqueness means an update changes one stored value. It does not
add another occurrence of the key. A missing key is a different case:
the specific `KeyError` handler below makes that result visible.


**Question.** After assigning `250` to ID `21`, how many pairs remain in bucket 1, and what value belongs to ID `11`?


In [2]:
study_minutes.put(21, 250)
print("Updated bucket:", study_minutes.buckets[1])
print("Pair count:", len(study_minutes.buckets[1]))
print("Minutes for ID 11:", study_minutes.get(11))
print("Minutes for ID 21:", study_minutes.get(21))

try:
    study_minutes.get(41)
except KeyError:
    print("ID 41 is not stored.")

Updated bucket: [(11, 100), (21, 250), (31, 300)]
Pair count: 3
Minutes for ID 11: 100
Minutes for ID 21: 250
ID 41 is not stored.


The bucket still has three pairs. ID `21` now maps to `250`, and ID `11`
still maps to `100`. ID `41` is absent even though it selects the same bucket.

With well-distributed hashes and enough buckets, expected lookup time is
`O(1)`. If many keys collide, searching a bucket can take `O(n)`. This
fixed-size demonstration deliberately puts every stored key in one bucket
so collision handling is visible.
